# Project SD-02 — PowerPoint RAG

> Goal: Build a RAG pipeline over a PowerPoint deck — where the changed block is
> the **parser**, not the model: shape-aware extraction, speaker notes, and one
> chunk per slide.

This is part of the **Special Documents** series (SD-01…SD-08). Every project in
the series swaps the *loading/parsing* stage of the baseline RAG pipeline and
keeps split → embed → store → retrieve → prompt → answer identical to
`04-baseline-rag.ipynb`. Here the document is not a web page or a PDF: it is a
`.pptx` slide deck, and most of the interesting information lives in *how* the
deck is structured.

**The stack:**

```
Parser      : python-pptx (per-slide, shape-aware)
Splitter    : none — one slide = one chunk
Embedding   : Gemini Embedding
Vector DB   : Chroma
Retriever   : Similarity Search (Top-K)
Prompt      : Basic Context + Question
LLM         : Gemini 2.5 Flash
```


## The naive way (what breaks)

Before building the "right" parser, let's look at what a naive extractor does —
this is the failure this project exists to fix. There are two traps.

**Trap 1 — flattening shapes.** A slide's *reading order* is defined by its
shape order, not by the order a text dump happens to hand you. A naive loop that
walks `slide.shapes` top-level only:

- never descends into **grouped shapes** (multiple shapes nested inside one
  group), so everything inside a group is silently lost;
- never treats the **title as the title** — it is just "a shape";
- and it completely ignores the **speaker notes slide**.

**Trap 2 — fixed-size chunking.** Even after you have text, a
`RecursiveCharacterTextSplitter` with a character budget does not know that a
slide is a self-contained unit of thought. It happily slices one slide's title,
bullets, and notes into unrelated chunks — severing the story mid-slide.

The fixture deck at `Data/SD-02-ppt/prs-notes.pptx` is deliberately
minimal (one slide, an empty grouped shape, a notes placeholder) so you can see
exactly what the naive loop finds — which is *nothing* — while a real deck makes
the same mistakes on every slide.

**WHAT TO EXPECT:** the first cell prints the slide count and an *empty* flat
text list; the second cell shows the presenter note that the naive loop dropped,
and a fixed-size splitter slicing a sample slide mid-bullet.


In [ ]:
from pptx import Presentation
from langchain_text_splitters import RecursiveCharacterTextSplitter

PPT_PATH = "../../Data/SD-02-ppt/prs-notes.pptx"

prs = Presentation(PPT_PATH)
print(f"slides in deck: {len(prs.slides)}")

for i, slide in enumerate(prs.slides, start=1):
    flat = [s.text for s in slide.shapes if s.has_text_frame]
    print(f"--- slide {i} naive flat text: {flat!r}")


In [ ]:
slide = prs.slides[0]

# What the naive loop silently dropped:
print("has_notes_slide:", slide.has_notes_slide)
print("notes (ignored):", repr(slide.notes_slide.notes_text_frame.text))

# Fixed-size chunking severs the slide as a unit:
sample = (
    "Title: Why PowerPoint RAG is hard\n"
    "- Shape order defines reading order\n"
    "- Speaker notes hold the real context\n"
)
naive_splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=10)
for j, ch in enumerate(naive_splitter.split_text(sample)):
    print(f"chunk {j}: {ch!r}")


## 0 · Setup — environment & keys

**WHAT:** Loads `.env` so the Gemini API key is available, then imports the
LangChain classes plus `python-pptx` (`Presentation`) for reading the deck.

**WHY:** The key check is *masked* — we print only `key[:4]…` to prove the key
is present, never the key itself. No install cell is needed here:
`python-pptx` is already in `requirements.txt` (unlike sentence-transformers or
faiss in other projects), so the parser dependency is ready out of the box.

**WHAT TO EXPECT:** `load_dotenv()` returns `True`, the masked-key line prints,
and the imports succeed. If you see "No GOOGLE_API_KEY found", create a `.env`
from `.env.example` and paste your key.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("GOOGLE_API_KEY")
if key:
    print(f"GOOGLE_API_KEY set (masked): {key[:4]}…")
else:
    print("No GOOGLE_API_KEY found — copy .env.example to .env and add yours.")


**WHAT these imports are for:**

| import | role in this project |
|---|---|
| `dotenv.load_dotenv` | read the API key from `.env` |
| `langchain_text_splitters.RecursiveCharacterTextSplitter` | imported so we can *reject* it — fixed-size chunking is the trap (Section 2) |
| `langchain_google_genai.GoogleGenerativeAIEmbeddings` | slide text → vectors |
| `langchain_google_genai.ChatGoogleGenerativeAI` | the answering LLM |
| `langchain_chroma.Chroma` | the vector store |
| `langchain_core.prompts.ChatPromptTemplate` | context + question wrapper |
| `langchain_core.documents.Document` | the slide-as-chunk container |
| `pptx.Presentation` | the OOXML parser — this project's changed block |

The `Presentation` import is the whole point of SD-02: everything else is the
same pipeline you already know from Project 01.


In [ ]:
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from pptx import Presentation


## 1 · Load — parse a `.pptx` with python-pptx

**WHAT:** A `.pptx` file is a ZIP archive of OOXML parts — one XML part per
slide, plus layouts, masters, and the optional notes slides. `python-pptx`
turns that archive into Python objects: `Presentation` → `slides` → `shapes`,
where a shape is a text box, a placeholder, a picture, or a **group** of shapes.

**WHY it matters for RAG:** unlike a PDF page (Project 02), a slide has *two*
text streams — the on-slide content *and* the speaker notes the audience never
sees. Both are retrievable evidence, and both are invisible to a plain-text
`cat` approach. The slide's layout name also gives us a free `section` label for
every chunk.

**WHAT TO EXPECT:** a guard that prints a friendly error if the sample file is
missing, then a `Presentation` object reporting the slide count and the slide-1
layout name (`"Blank"` in this fixture).


In [ ]:
import os

if not os.path.exists(PPT_PATH):
    raise FileNotFoundError(
        "Sample deck not found. Expected at Data/SD-02-ppt/prs-notes.pptx."
    )

prs = Presentation(PPT_PATH)
print(f"slides in deck: {len(prs.slides)}")
print(f"slide 1 layout: {prs.slides[0].slide_layout.name!r}")


**Shape-aware extraction — WHAT + WHY.**

Three small helpers do the real parsing work:

1. `walk_shapes` — walks `slide.shapes` and **recursively descends** into
   grouped shapes (`shape.shape_type == MSO_SHAPE_TYPE.GROUP` → `shape.shapes`).
   Without this, everything inside a group is silently lost.
2. `extract_slide_text` — collects text in **shape order** and lifts the title
   placeholder to the front, so the chunk reads the way the audience read the
   slide.
3. `get_notes` — reads `slide.notes_slide.notes_text_frame.text` **guarded** by
   `slide.has_notes_slide`, so slides without notes never crash the parser.


In [ ]:
from pptx.enum.shapes import MSO_SHAPE_TYPE


def walk_shapes(shapes):
    """Yield (shape, depth), descending recursively into grouped shapes."""
    for shape in shapes:
        yield shape, 0
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            yield from walk_shapes(shape.shapes)


In [ ]:
def extract_slide_text(slide):
    """Title first, then body shapes in shape order (the reading order)."""
    title = slide.shapes.title
    lines = []
    for shape, _depth in walk_shapes(slide.shapes):
        if not shape.has_text_frame or not shape.text_frame.text.strip():
            continue
        lines.append(shape.text_frame.text.strip())
    if title is not None and title.has_text_frame and title.text_frame.text.strip():
        lines.insert(0, title.text_frame.text.strip())
    return list(dict.fromkeys(lines))


In [ ]:
def get_notes(slide):
    """Guarded speaker-note access — '' if the slide has no notes."""
    if slide.has_notes_slide:
        return slide.notes_slide.notes_text_frame.text.strip()
    return ""


In [ ]:
def load_slides(path):
    """One Document per slide; notes mirrored into text AND metadata."""
    prs = Presentation(path)
    docs = []
    for i, slide in enumerate(prs.slides, start=1):
        lines = extract_slide_text(slide)
        notes = get_notes(slide)
        if notes:
            lines.append(f"[Speaker notes] {notes}")
        meta = {"slide_number": i, "section": slide.slide_layout.name or "General", "notes": notes}
        docs.append(Document(page_content="\n".join(lines), metadata=meta))
    return docs


In [ ]:
docs = load_slides(PPT_PATH)
print(f"documents (one per slide): {len(docs)}")

doc = docs[0]
print(f"metadata: {doc.metadata}")
print(f"content : {doc.page_content!r}")


## 2 · Split — the slide is the atomic unit

**WHAT:** Zero re-splitting. Our loader already returns one `Document` per
slide, so "chunking" is a passthrough — `chunks = docs`.

**WHY:** A slide is the natural unit of thought in a deck: one idea, one layout,
one reading order. The `RecursiveCharacterTextSplitter` from the baseline is the
wrong tool here because it splits by *character budget*, not by *meaning* — a
long bullet can be sliced mid-sentence (you saw this in the naive demo). Keeping
the slide whole also preserves the title + notes pairing that makes slide
retrieval work.

**WHAT TO EXPECT:** the same document count as load, plus a short demo showing
that a fixed-size splitter would fragment a longer slide into unrelated pieces.


In [ ]:
chunks = docs  # one Document per slide → the slide IS the chunk
print(f"chunks (slides): {len(chunks)}")
for ch in chunks:
    print(f"  slide {ch.metadata['slide_number']} · {ch.page_content[:50]!r}")


In [ ]:
# The trap: applying a fixed-size splitter to slide text fragments the story.
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
for ch in chunks:
    pieces = splitter.split_text(ch.page_content)
    print(f"slide {ch.metadata['slide_number']}: {len(pieces)} fixed-size piece(s)")
print("→ this deck's slides are short, but real 400-char slides get severed.")


## 3 · Embed — slide text → vectors

**WHAT:** `GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")`
turns each slide's text into a vector — numbers that put similar text nearby in
vector space.

**WHY:** Retrieval is a vector-space search. Because our chunk is a *whole
slide*, the embedding represents the slide's complete idea (title + body +
notes) rather than an arbitrary 1000-character slice — so a query like
"Summarize slide 1." can find the slide as a whole.

**WHAT TO EXPECT:** an embeddings object, then a demo embedding showing its
dimensions — a long dense vector of floats.


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")


In [ ]:
vec = embeddings.embed_query("speaker notes about delivery")
print(f"embedding dimensions: {len(vec)}")
print(f"first 5 values      : {vec[:5]}")


## 4 · Store — index the slide chunks in Chroma

**WHAT:** `Chroma.from_documents(documents=chunks, embedding=embeddings)`
embeds every slide chunk and indexes the vectors in a Chroma collection.

**WHY:** The vector store is the pipeline's memory. Embedding happens *here* —
every slide becomes a searchable point — and Chroma persists the index so the
retriever can query it later.

**WHAT TO EXPECT:** a `Chroma` object, and a `chroma_langchain_db/` directory
written next to the notebook (a regenerable artifact, gitignored).


In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)


## 5 · Retrieve — top-k slide search

**WHAT:** `vector_store.similarity_search(query, k=3)` finds the 3 chunks
(slides) whose vectors are nearest to the query vector. We use the QA pair
**"Summarize slide 1."** to make the retrieval concrete.

**WHY:** This is the "R" in RAG. Because each chunk carries `slide_number` and
`section` metadata, the retriever tells us *which slide* the evidence came from
— and we can even scope the search to a single slide with a metadata filter
(you'll try that in Section 8).

**WHAT TO EXPECT:** the slide-1 chunk returned as the top hit, and the second
cell printing each hit with its relevance score and `slide_number`.


In [ ]:
query = "Summarize slide 1."

hits = vector_store.similarity_search(query=query, k=3)
print(f"retrieved {len(hits)} chunk(s) for {query!r}")


In [ ]:
results = vector_store.similarity_search_with_relevance_scores(query=query, k=3)
for doc, score in results:
    meta = doc.metadata
    print(f"[{score:.3f}] slide {meta['slide_number']} ({meta['section']!r}): {doc.page_content[:60]!r}")


## 6 · Prompt — package context + question

**WHAT:** The same `ChatPromptTemplate` as the baseline: instructions
("answer using ONLY the provided context", with an "I don't know" fallback)
wrapped around `{context}` and `{question}` slots.

**WHY:** Grounding. The template forbids the model from answering from memory —
it must answer from the retrieved slide chunks. That is what turns the LLM into
a *deck-aware* assistant. Reading the rendered `messages` shows exactly what the
model receives.

**WHAT TO EXPECT:** a `ChatPromptTemplate`, then a printed `messages` object
with the instructions, the retrieved slide text as context, and the query.


In [ ]:
template = """You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)


In [ ]:
context = "\n\n".join(doc.page_content for doc in hits)

messages = prompt.invoke({"context": context, "question": query})
print(messages)


## 7 · Answer — the LLM reads the prompt

**WHAT:** `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` takes the filled
`messages` and produces the answer, which is printed.

**WHY:** The final block. The model summarizes the *retrieved slide*, not a
guess — the "Summarize slide 1." query demonstrates that the chunk boundary (the
slide unit) directly shapes how the model can answer.

**WHAT TO EXPECT:** a natural-language summary of the slide grounded in the
retrieved chunk (in this fixture, the notes placeholder text).


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
response = llm.invoke(messages)
print(response.content)


## 8 · Try it yourself — your sandbox

Two extra query cells: the second QA pair, **"What did the presenter note about
this deck?"** — which specifically exercises the *notes* that the naive parser
dropped — and a **metadata-scoped** retrieval that restricts search to a single
slide. Change `k`, change `filter`, or write a completely new question and watch
the answers shift.


In [ ]:
# QA pair #2: notes-focused retrieval.
query2 = "What did the presenter note about this deck?"
hits2 = vector_store.similarity_search(query=query2, k=3)

for hit in hits2:
    print(f"slide {hit.metadata['slide_number']} | notes={hit.metadata['notes']!r}")

context2 = "\n\n".join(doc.page_content for doc in hits2)
messages2 = prompt.invoke({"context": context2, "question": query2})
response2 = llm.invoke(messages2)
print(response2.content)


In [ ]:
# Scoped retrieval: restrict to one slide using its metadata.
scoped = vector_store.similarity_search(
    query="Summarize slide 1.",
    k=3,
    filter={"slide_number": 1},
)
print(f"scoped hits: {len(scoped)}")
for hit in scoped:
    print(f"  slide {hit.metadata['slide_number']}: {hit.page_content[:60]!r}")


## What you should notice

- **Parsing, not the model, is the changed block.** The pipeline after Load is
  the same baseline; retrieval quality is decided entirely by how we turned
  slides into chunks.
- **Reading order lives in shape order.** A naive `for shape in slide.shapes`
  flattening scrambles meaning; the parser must honour title-first, then body
  shapes in order.
- **Groups hide content.** `shape.shape_type == MSO_SHAPE_TYPE.GROUP` means
  there are shapes *inside* the shape — recursion (`shape.shapes`) is
  non-optional on real decks.
- **Speaker notes are half the evidence.** They are invisible to plain-text
  extraction and absent from `slide.shapes` — only `notes_slide` surfaces them,
  and they answer a whole class of queries ("what did the presenter note…").
- **The slide is the atomic chunk.** A fixed-size `RecursiveCharacterTextSplitter`
  severs a slide's story; one `Document` per slide keeps title + bullets + notes
  together as one retrievable unit.
- **`.pptx` is a ZIP.** Knowing the OOXML structure (slides + layouts + notes
  parts) is what lets `python-pptx` give us `slide_number` and `section` for
  free — metadata that plain text never had.


## Exercises

1. **Extract tables.** Extend `walk_shapes` (or `extract_slide_text`) to handle
   `shape.has_table` — join each table's `row.cells` into a line and add it to
   the slide content. Why does a naive walker skip tables entirely?
2. **Enrich metadata.** Real decks carry named sections
   (`prs.sectioned_slides`) and picture alt text (`shape.alt_text`). Map each
   slide to its *named section* instead of the layout name, add picture alt text
   as content, then re-run Section 5 with a `filter={"section": ...}` query.
3. **Compare with Project 02.** Both projects are "parse a document type".
   What is different about chunking a *slide* versus a *PDF page*? Which deck
   structures (title + notes metadata) have no equivalent in PDFs?
